# Netflix Content Analysis: An Exploratory Data Analysis

**Project type:** Final Data Science Internship Project  
**Dataset:** Netflix Movies and TV Shows catalog dataset  
**Analysis focus:** content composition, growth, genres, geography, ratings, duration, and release/addition timing.

> **Important:** This analysis describes the titles represented in the supplied dataset. It should not be interpreted as a live snapshot of Netflix's catalog in 2026.

## Research Problem

Netflix's catalog contains movies and TV shows from many countries, genres, release periods, and rating categories. The goal of this project is to identify meaningful patterns in the supplied catalog and understand how content type, geography, genre, ratings, duration, and timing are distributed.

## Main Research Question

**What patterns and trends can be identified in the Netflix catalog based on content type, release year, genres, countries, ratings, duration, and the timing of content addition?**

### Supporting Questions
1. How is the catalog distributed between Movies and TV Shows?
2. How did title additions vary across the years represented in the dataset?
3. Which genres/categories are most prevalent?
4. Which countries contribute the most title associations?
5. How do ratings differ between Movies and TV Shows?
6. What does the duration structure look like for Movies and TV Shows?
7. What relationship exists between original release year and Netflix addition year?

## Methodology
The workflow follows: **data loading → exploration → quality assessment → cleaning → feature engineering → univariate/bivariate analysis → visualization → findings → limitations**.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)


## 2. Load the Dataset

In [ ]:
DATA_PATH = Path("../data/netflix_titles.csv")
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
display(df.head())


## 3. Initial Exploration

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.info())
display(df.describe(include="all").T)


In [ ]:
print("Missing values:")
display(df.isna().sum().sort_values(ascending=False))

print("Duplicate rows:", df.duplicated().sum())
print("Unique titles:", df["title"].nunique())


## 4. Data Quality Assessment

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean()*100).round(2),
    "unique_values": df.nunique()
}).sort_values("missing_pct", ascending=False)

display(quality)


### Cleaning decisions

- Convert `date_added` to a proper datetime field.
- Preserve missing categorical information as `Unknown` rather than dropping large portions of the catalog.
- Extract numeric duration values from strings such as `90 min` and `3 Seasons`.
- Create year/month fields from `date_added`.
- Split multi-valued `listed_in` and `country` fields for category-level analysis.
- Keep the raw dataset unchanged for reproducibility.


## 5. Data Cleaning and Feature Engineering

In [ ]:
clean = df.copy()

clean["date_added"] = pd.to_datetime(clean["date_added"], errors="coerce")

for col in ["director", "cast", "country", "rating"]:
    clean[col] = clean[col].fillna("Unknown")

clean["year_added"] = clean["date_added"].dt.year
clean["month_added"] = clean["date_added"].dt.month
clean["month_name"] = clean["date_added"].dt.month_name()

clean["duration_value"] = pd.to_numeric(
    clean["duration"].str.extract(r"(\d+)")[0],
    errors="coerce"
)
clean["duration_unit"] = clean["duration"].str.extract(r"([A-Za-z]+)$")[0]

clean["content_age_at_addition"] = clean["year_added"] - clean["release_year"]
clean["content_age_at_addition"] = clean["content_age_at_addition"].where(
    clean["content_age_at_addition"] >= 0
)

display(clean.head())


## 6. Overall Catalog Composition

In [ ]:
type_counts = clean["type"].value_counts()
display(type_counts.to_frame("count"))

fig, ax = plt.subplots(figsize=(8,5))
type_counts.plot(kind="bar", ax=ax)
ax.set_title("Netflix Catalog by Content Type")
ax.set_xlabel("")
ax.set_ylabel("Number of Titles")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


### Interpretation

The chart above answers the first supporting question by showing the relative representation of Movies and TV Shows in the supplied dataset. The exact counts should be read directly from the executed output rather than assumed in advance.


## 7. Content Addition Trends Over Time

In [ ]:
added_by_year = (
    clean.dropna(subset=["year_added"])
         .groupby(["year_added", "type"])
         .size()
         .unstack(fill_value=0)
)

display(added_by_year)

fig, ax = plt.subplots(figsize=(11,6))
added_by_year.plot(marker="o", ax=ax)
ax.set_title("Netflix Titles Added by Year")
ax.set_xlabel("Year Added")
ax.set_ylabel("Number of Titles")
ax.legend(title="Content Type")
plt.tight_layout()
plt.show()


## 8. Genre / Category Analysis

In [ ]:
genre_df = clean.assign(
    genre=clean["listed_in"].str.split(", ")
).explode("genre")

top_genres = genre_df["genre"].value_counts().head(15)
display(top_genres.to_frame("title_category_associations"))

fig, ax = plt.subplots(figsize=(10,7))
top_genres.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top 15 Genres / Categories")
ax.set_xlabel("Number of Title-Category Associations")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
genre_type = pd.crosstab(genre_df["genre"], genre_df["type"])
top_genre_names = top_genres.head(10).index
display(genre_type.loc[top_genre_names])


## 9. Geographic Analysis

In [ ]:
country_df = clean.assign(
    country_clean=clean["country"].str.split(", ")
).explode("country_clean")

country_counts = (
    country_df.loc[country_df["country_clean"] != "Unknown", "country_clean"]
              .value_counts()
              .head(15)
)

display(country_counts.to_frame("title_country_associations"))

fig, ax = plt.subplots(figsize=(10,7))
country_counts.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top 15 Countries by Title Associations")
ax.set_xlabel("Number of Title-Country Associations")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


**Interpretation note:** A title can have multiple countries, so these are title-country associations, not mutually exclusive title counts.


## 10. Rating Analysis

In [ ]:
rating_type = pd.crosstab(clean["rating"], clean["type"])
top_ratings = clean["rating"].value_counts().head(12).index

display(rating_type.loc[top_ratings])

fig, ax = plt.subplots(figsize=(11,6))
rating_type.loc[top_ratings].plot(kind="bar", ax=ax)
ax.set_title("Content Ratings by Content Type")
ax.set_xlabel("Rating")
ax.set_ylabel("Number of Titles")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 11. Duration and Season Analysis

In [ ]:
movies = clean[clean["type"] == "Movie"].copy()
tv_shows = clean[clean["type"] == "TV Show"].copy()

movie_duration = movies["duration_value"].dropna()
tv_seasons = tv_shows["duration_value"].dropna()

print("Movie duration summary:")
display(movie_duration.describe().to_frame("minutes"))

print("TV season summary:")
display(tv_seasons.describe().to_frame("seasons"))


In [ ]:
fig, ax = plt.subplots(figsize=(10,6))
movie_duration.plot(kind="hist", bins=30, ax=ax)
ax.set_title("Distribution of Movie Durations")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Number of Movies")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10,6))
tv_seasons.plot(kind="hist", bins=15, ax=ax)
ax.set_title("Distribution of TV Show Season Counts")
ax.set_xlabel("Number of Seasons")
ax.set_ylabel("Number of TV Shows")
plt.tight_layout()
plt.show()


## 12. Release Year vs Netflix Addition Year

In [ ]:
release_added = clean.dropna(subset=["release_year", "year_added"]).copy()

fig, ax = plt.subplots(figsize=(10,6))
sns.scatterplot(
    data=release_added.sample(min(2500, len(release_added)), random_state=42),
    x="release_year",
    y="year_added",
    hue="type",
    alpha=0.45,
    ax=ax
)
ax.set_title("Original Release Year vs Netflix Addition Year")
ax.set_xlabel("Original Release Year")
ax.set_ylabel("Year Added to Netflix")
plt.tight_layout()
plt.show()


### Content age at addition
The engineered `content_age_at_addition` field measures the gap between original release year and Netflix addition year. A larger value indicates older content, relative to the year it entered the dataset's Netflix catalog.


In [ ]:
age_by_type = (
    clean.dropna(subset=["content_age_at_addition"])
         .groupby("type")["content_age_at_addition"]
         .agg(["count","mean","median","min","max"])
)

display(age_by_type)


## 13. Advanced Temporal Analysis: Year × Month Heatmap

In [ ]:
month_year = pd.crosstab(clean["year_added"], clean["month_added"])

fig, ax = plt.subplots(figsize=(12,6))
sns.heatmap(month_year, cmap="Blues", ax=ax)
ax.set_title("Titles Added by Year and Month")
ax.set_xlabel("Month")
ax.set_ylabel("Year")
plt.tight_layout()
plt.show()


## 14. Key Findings

In [ ]:
# Generate a compact set of data-driven metrics for the conclusion.
movie_count = int((clean["type"] == "Movie").sum())
tv_count = int((clean["type"] == "TV Show").sum())
movie_share = movie_count / len(clean) * 100
tv_share = tv_count / len(clean) * 100

top_genre = genre_df["genre"].value_counts().idxmax()
top_country = country_df.loc[
    country_df["country_clean"] != "Unknown", "country_clean"
].value_counts().idxmax()
top_rating = clean["rating"].value_counts().idxmax()

print(f"Total titles: {len(clean):,}")
print(f"Movies: {movie_count:,} ({movie_share:.1f}%)")
print(f"TV Shows: {tv_count:,} ({tv_share:.1f}%)")
print(f"Most frequent category: {top_genre}")
print(f"Most frequent country association: {top_country}")
print(f"Most frequent rating: {top_rating}")


## 15. Conclusion

This analysis uses exploratory data analysis and visualization to examine the supplied Netflix catalog dataset across content type, temporal additions, genres, countries, ratings, duration, and release/addition timing.

The final conclusions should be based on the executed outputs above. In particular, the analysis demonstrates how transforming multi-valued fields, extracting temporal features, and separating Movies from TV Shows enables more meaningful comparisons.

### Answer to the Research Question

The executed visualizations collectively show the structure and major patterns of the catalog represented in the dataset. They provide evidence for differences in content type, category prevalence, geographic representation, ratings, duration structure, and the timing relationship between original release and Netflix addition.


## 16. Limitations

1. The dataset is a historical catalog snapshot, not a live 2026 Netflix catalog.
2. It does not provide viewer watch counts, revenue, engagement, or popularity measures.
3. `date_added` indicates catalog addition, not production or original release.
4. Multi-valued country and genre fields mean counts can represent associations rather than mutually exclusive titles.
5. Missing metadata is present for several columns.
6. The dataset alone cannot establish causal relationships.

## 17. Future Scope

- Natural-language processing of descriptions.
- Sentiment or thematic analysis of title descriptions.
- Recommendation-system development.
- Clustering titles by metadata/features.
- Predictive modeling for content characteristics.
- Interactive dashboards using Plotly or Power BI.


## 18. Reproducibility

The repository keeps the raw dataset separate from the cleaned dataset. All transformations are implemented in the notebook, and the exported CSV/PNG files in `outputs/` provide reusable artifacts.

**Run order:** execute the notebook from top to bottom.
